In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [2]:
DETECTORS = ["AD11", "AD12", "AD21", "AD22", "AD31", "AD32", "AD33", "AD34"]

def get_events():
    base_path = "dayabay_full_dataset_npz_1-0-0/npz/dayabay_dataset"

    events = {}
    for detector in DETECTORS:
        data = np.load(f"{base_path}/dayabay_events_{detector}.npz")
        events[detector] = data["events"]
        data.close()

    return events

In [3]:
events = get_events()

In [ ]:
# из вектора получить единичный
unit = lambda x: x/np.linalg.norm(x)

Будем считать, что позитрон перед уничтожением пройдёт слишком малое расстояние, поэтому координата мгновенного (prompt) сигнала это примерно координата, в которой произошёл IBD (в статье я не нашёл лучшего метода)

Из этого предположения найти пространственное распределение получаемых позитронов очевидно невозможно. Однако можно найти пространственное распределение испускаемых нейтронов (вполне логично считать, что координата delayed сигнала это и есть конечная координата нейтрона)

Будем считать азимутальный угол (в цилиндрических координатах детектора) конечного положения нейтрона. Так как траектория нейтрона необязательно прямая, то для одного события посчитанный угол не будет определять исходное направление движения нейтрона (сразу после IBD). Но для большого числа событий, в среднем, это, _предположу_, верно

Для каждого детектора найдём "среднее" направление протекания IBD (взвешенное (т.е., получив массив векторов dr от delayed- к prompt-сигналу, сначала найти среднее, а потом сделать из этого единичный вектор) и невзвешенное (для каждого dr сначала посчитать единичный вектор от delayed- к prompt-сигналу, и только затем усреднить))

Умная статья https://arxiv.org/pdf/2406.08359 говорит, что (в моих терминах) "взвешенное" усреднение даёт направление исходного антинейтрино (вектор направлен от "реактора" до детектора)

Из нашей статьи найдём радиус-вектора ко всем детекторам/реакторам. Вычтя из одного другое найдём радиус-вектор от реактора к детектору (в координатах которые в статье даны). Если надо, как-то пересчитаем это в цилиндрические координаты каждого детектора(как?). Эти вектора сопоставим с полученными выше средними (если они ненулевые)

In [29]:
def get_detector_plot(detector):
    r_p = events[detector][["vertex_prompt_x_mm", "vertex_prompt_y_mm", "vertex_prompt_z_mm"]]
    x_p = r_p["vertex_prompt_x_mm"]
    y_p = r_p["vertex_prompt_y_mm"]
    z_p = r_p["vertex_prompt_z_mm"]

    r_d = events[detector][["vertex_delayed_x_mm", "vertex_delayed_y_mm", "vertex_delayed_z_mm"]]
    x_d = r_d["vertex_delayed_x_mm"]
    y_d = r_d["vertex_delayed_y_mm"]
    z_d = r_d["vertex_delayed_z_mm"]

    dx = x_d - x_p
    dy = y_d - y_p
    dz = z_d - z_p
    l = np.sqrt(dx**2 + dy**2 + dz**2)

    # радиус вектор (предполагаемого) направления протекания IBD
    dr = np.vstack((dx, dy, dz)).T
    # единичный вектор направления IBD
    u = np.vstack((dx/l, dy/l, dz/l)).T

    # polar
    theta = np.acos(dz/l)
    # azimuthal angle for each dr
    phi = np.arctan2(dy, dx)
    
    # азимутальный угол усредненного единичного вектора 
    phi1 = np.arctan2(u.mean(axis=0)[1], u.mean(axis=0)[0])

    print(f"\n{'=' * 45}")
    print(f"Statistics for {detector}")
    print(f"{'=' * 45}")

    print(f"  Unit vector of mean dR:      {unit(dr.mean(axis=0))}")
    print(f"  Mean of unit vectors:        {unit(u.mean(axis=0))}")
    print(f"  Average polar angle:         {(theta.mean()):8.3f}")
    print(f"  Average azimuthal angle:     {(phi.mean()):8.3f}")
    print(f"  Azimuth of mean unit vector: {(phi1) :8.3f}")

    print(f"{'=' * 45}\n")

    # plt.show()

In [30]:
# радиус-вектора к реакторам/детекторам взяты из таблицы в статье
reactors = np.load("dayabay_reactors.npz")
detectors = np.load("dayabay_detectors.npz")
R = reactors["reactor_xyz_m"]
D = detectors["detector_xyz_m"]


for i, detector in enumerate(DETECTORS):
    get_detector_plot(detector)

    # d = D[i] - R[i]
    # радиус-вектор от детектора к реактору, но в другой координатной системе, это надо пересчитать (как-то)




Statistics for AD11
  Unit vector of mean dR:      [ 0.7877911  -0.52451205  0.32291237]
  Mean of unit vectors:        [ 0.81437236 -0.5788112   0.04213403]
  Average polar angle:            1.569
  Average azimuthal angle:       -0.041
  Azimuth of mean unit vector:   -0.618


Statistics for AD12
  Unit vector of mean dR:      [ 0.74661505 -0.60844696  0.268995  ]
  Mean of unit vectors:        [ 0.80070066 -0.598425    0.0276769 ]
  Average polar angle:            1.570
  Average azimuthal angle:       -0.046
  Azimuth of mean unit vector:   -0.642


Statistics for AD21
  Unit vector of mean dR:      [-0.70389485 -0.68383366  0.1921028 ]
  Mean of unit vectors:        [-0.7376261  -0.67454803  0.02987854]
  Average polar angle:            1.569
  Average azimuthal angle:       -0.062
  Azimuth of mean unit vector:   -2.401


Statistics for AD22
  Unit vector of mean dR:      [-0.754733   -0.6311215   0.17906313]
  Mean of unit vectors:        [-0.77072465 -0.63678616  0.02206541]
 

Как видим, два усреднения дали слишком разные результаты

1.Для обоих посчитанных углов видно: для близких детекторов углы примерно равны, а для далёких они различаются

2.Углы ненулевые, т.е. у IBD есть направленность. Причем из (1) она, предположительно, зависит от географического положения детектора относительно реакторов